# Introduction to MODFLOW 6 and MODPATH 7: A Regional Alluvial Aquifer

**Instructor:** Jesus D. Gomez-Velez (jesus-gomezvelez@uiowa.edu)

MODFLOW 6 is the latest version of the U.S. Geological Survey's modular groundwater flow model. It simulates groundwater movement in aquifers and other subsurface systems by solving equations that describe water storage and flow under a wide range of hydrogeologic conditions. In an introductory groundwater course, MODFLOW 6 is valuable because it connects fundamental concepts (hydraulic head, Darcy's law, conservation of mass, aquifer properties, and boundary conditions) to a practical computational framework. It is widely used to evaluate pumping, recharge, stream-aquifer interaction, and the effects of natural and human stresses on groundwater systems.

MODFLOW 6 also suits teaching because of its modular architecture: each process (aquifer properties, rivers, wells, recharge, and so on) is a separate *package*, and the packages are combined to build a model. You can see how a groundwater system is divided into cells, how the governing equations are assembled for each cell, and how numerical methods estimate heads and fluxes. Starting with a simple example and then adding complexity shows how model assumptions, parameter values, and boundary conditions control groundwater behavior.

In this notebook, we build and run a MODFLOW 6 model from Python using **FloPy**, a Python package developed by the USGS and collaborators to create, run, and post-process MODFLOW models. We then use **MODPATH 7**, a particle-tracking program, to find where the water pumped by a well comes from (its *capture zone*).

## Learning Objectives

By the end of this notebook, you should be able to:

1. Translate a **conceptual model** of an aquifer (geometry, properties, boundaries, and stresses) into a numerical MODFLOW 6 model.
2. Explain the role of each MODFLOW 6 package used here (TDIS, IMS, DIS, NPF, IC, RIV, CHD, RCH, WEL, OC).
3. Distinguish between **specified-head**, **specified-flux** (including no-flow), and **head-dependent flux** boundary conditions.
4. Run MODFLOW 6 from Python with **FloPy**, and read and plot heads, flow vectors, and water-table depth.
5. Identify **gaining** and **losing** river reaches and explain how pumping changes stream–aquifer exchange.
6. Use **MODPATH 7** backward particle tracking to delineate a well **capture zone** and identify the sources of water to a well.

## The Problem: A Regional Alluvial Aquifer Next to a River

We model an idealized alluvial valley, 10 km wide (west–east) and 20 km long (north–south), drained by a river that flows south along its eastern edge. A municipal well pumps water from the middle of the valley. We want to answer:

- What does the water table look like, and in which direction does groundwater flow?
- Does the river gain water from the aquifer or lose water to it?
- Where does the water pumped by the well come from: regional groundwater or the river?

**Plan view of the conceptual model** (not to scale):

```
                         North  (y = 20,000 m)  — upstream
                 CHD: h = 109.95 m along the entire first row
        x = 0  ┌──────────────────────────────────────────┐  x = 10,000 m
               │                                      │ R │
               │                     ⊕ W2 (off)       │ I │
     No-flow   │                                      │ V │
     boundary  │                            ● W1 (off)│ E │   River (RIV)
     (valley   │                                      │ R │   flows south;
      wall)    │                                      │   │   stage drops
               │                  ● W3 (pumping)      │   │   from 109.95 m
               │                                      │   │   to 100.05 m
               │                                      │   │
               │                                      │   │
               └──────────────────────────────────────────┘
                 CHD: h = 100.05 m along the entire last row
                         South  (y = 0 m)  — downstream
```

**Summary of model inputs:**

| Component | Value | Package |
|---|---|---|
| Domain | 10 km × 20 km, 1 layer, 100 m thick | DIS |
| Grid | 100 rows × 50 columns, 200 m × 200 m cells | DIS |
| Land surface | Slopes south at 0.0005 (≈ 100–110 m) | DIS |
| Hydraulic conductivity | $K$ = 15 m/d, unconfined | NPF |
| River | East column, stage 109.95 → 100.05 m, $C_{riv}$ = 500 m²/d | RIV |
| Regional boundaries | Constant head on the north and south rows | CHD |
| Recharge | 0.0001 m/d (switched **off** in the base case) | RCH |
| Wells | Well 3 pumps 10,000 m³/d; Wells 1 and 2 are switched **off** | WEL |
| Effective porosity | 0.25 (particle tracking only) | MODPATH |

**Key assumptions:** steady state, two-dimensional horizontal flow (Dupuit approximation), homogeneous and isotropic aquifer, and an impermeable base.

### Notation

| Symbol | Meaning | Units |
|---|---|---|
| $h$ | hydraulic head (= water-table elevation in an unconfined aquifer) | m |
| $z_{top}$, $z_{bot}$ | top and bottom elevation of the aquifer | m |
| $b = h - z_{bot}$ | saturated thickness | m |
| $K$ | hydraulic conductivity | m/d |
| $T = K\,b$ | transmissivity | m²/d |
| $R$ | recharge rate | m/d |
| $Q$ | volumetric flow rate (wells, river leakage) | m³/d |
| $\mathbf{q}$ | specific discharge (Darcy flux) | m/d |
| $n_e$ | effective porosity | – |
| $\mathbf{v} = \mathbf{q}/n_e$ | average seepage (linear) velocity | m/d |

## Step 0. Load the Required Libraries and Locate the Executables

We use four Python libraries:

| Library | Purpose |
|---|---|
| `os` | Builds file paths that work on any operating system |
| `numpy` | Arrays and numerical operations (grids, elevations, heads) |
| `matplotlib` | Plotting |
| `flopy` | Writes MODFLOW/MODPATH input files, runs the programs, and reads their output |

FloPy **does not** solve the groundwater flow equation itself. It writes text input files and then calls the compiled MODFLOW 6 (`mf6`) and MODPATH 7 (`mp7`) programs. For that reason, we must tell Python where those executables are.

Set `MODFLOW_BIN_DIR` to the folder that contains the executables on **your** computer. For example, if you downloaded the binaries and extracted them to a folder named `usgs_code` in your current working directory, set `MODFLOW_BIN_DIR = "./usgs_code"`.

> **Tip:** The easiest way to get the USGS executables is FloPy's installer. Run `get-modflow :flopy` in a terminal, or run `flopy.utils.get_modflow(":flopy")` in Python. It installs them in `~/.local/share/flopy/bin`, which is the folder used below. If you compiled the codes yourself (for example, on IDAS), point to that folder instead.

In [ ]:
import os # Operating system library
import numpy as np # Numerical library
import matplotlib.pyplot as plt # Plotting library
import flopy # Calls modflow and other USGS programs
from flopy.plot import PlotMapView # Plotting library for MODFLOW models

MODFLOW_BIN_DIR = "/Users/jgomezvelez/.local/share/flopy/bin"
MF6_EXE = os.path.join(MODFLOW_BIN_DIR, "mf6")
MP7_EXE = os.path.join(MODFLOW_BIN_DIR, "mp7")

## Step 1. Initialize the Simulation, Time Discretization, and Solver

A MODFLOW 6 run is organized as a hierarchy of objects:

```
Simulation (MFSimulation)  ← overall container; writes the files and runs mf6
├── TDIS   time discretization: stress periods and time steps
├── IMS    iterative solver settings
└── GWF model ("alluvial_model")  ← created in Step 2
    ├── DIS  spatial grid                 (Step 3)
    ├── NPF  hydraulic conductivity       (Step 4)
    ├── IC   initial heads                (Step 4)
    ├── RIV  river boundary               (Step 5)
    ├── CHD  constant-head boundary       (Step 5)
    ├── RCH  recharge                     (Step 6)
    ├── WEL  wells                        (Step 7)
    └── OC   output control               (Step 8)
```

In this step, we create the top three pieces:

1. **`MFSimulation`** sets the simulation name, the path to the `mf6` executable, and the *workspace* (`sim_ws`), the folder where all input and output files are written. Open `workspace_alluvial_model/` after Step 9 to see the files FloPy creates.

2. **`ModflowTdis`** (time discretization) defines the **stress periods**. A stress period is an interval during which the boundary conditions and stresses (pumping, recharge, river stage) stay constant. Each period has a length, a number of time steps, and a time-step multiplier. Here we use one stress period of 1 day with one time step.

   *Why does a "1-day" period give a steady-state answer?* A model is transient only if it includes a storage package (STO) that declares a period transient. This model has no STO package, so MODFLOW 6 treats the period as **steady state**: heads do not change with time, and the period length has no effect on the result.

3. **`ModflowIms`** (Iterative Model Solution) controls how MODFLOW solves the system of equations: convergence tolerances, maximum iterations, and the linear solver. `complexity="MODERATE"` selects a preset set of solver options. It describes the *solver configuration*, not the physical complexity of the aquifer. Our model is unconfined, so the equations are **nonlinear** (transmissivity depends on the unknown head; see Step 4). For that reason, the solver repeats its calculations ("outer iterations") until the heads stop changing.

> **Units:** MODFLOW does not convert units. You must use one consistent set throughout. Here we use **meters** for length and **days** for time, so K is in m/d, recharge is in m/d, and pumping rates are in m³/d.

In [ ]:
# Model workspace and names
model_dir = "./workspace_alluvial_model"
model_name = "alluvial_model"

# Create simulation: This creates the main MODFLOW 6 simulation object and assigns it to sim. It is the container to which you’ll add the groundwater model and its packages.
sim = flopy.mf6.MFSimulation(
    sim_name=model_name, 
    version="mf6", 
    exe_name=MF6_EXE, 
    sim_ws=model_dir
)

# Define Time Discretization (TDIS) - 1 steady-state stress period
tdis = flopy.mf6.ModflowTdis(
    sim, # attaches it to the MODFLOW simulation
    nper=1, # sets one stress period.
    perioddata=[(1.0, 1, 1.0)],  # (length of period, number of steps, time step multiplier)
    time_units="DAYS"
)

# Define iterative model solver (IMS): tells MODFLOW how to solve the model’s equations.
ims = flopy.mf6.ModflowIms( 
    sim, # attaches that solver package to the simulation
    pname="ims", # gives the package the name ims.
    complexity="MODERATE" # selects FloPy/MODFLOW’s moderate preset for solver settings, such as how tightly and how quickly the solver seeks a solution. It describes the solver configuration—not the physical complexity of the groundwater model.
)

## Step 2. Create the Groundwater Flow (GWF) Model

Next, we create the groundwater-flow model object (`gwf`), the MODFLOW 6 component that represents the aquifer. It is attached to the simulation created in Step 1 and named `alluvial_model`. FloPy also creates a *name file* (`alluvial_model.nam`), which lists every package that belongs to this model. MODFLOW reads it to find the rest of the input files.

At this point, the model is an empty container. We have not yet specified the aquifer geometry, hydraulic properties, or boundary conditions. In the following steps, we add one package at a time to `gwf`.

> **Why separate "simulation" and "model"?** A single MODFLOW 6 simulation can contain several models that run together, for example a flow model (GWF) coupled with a solute-transport model (GWT), or two flow models with different grids. In this course we use one GWF model, but the structure is the same for more advanced applications.

In [ ]:
# Create the groundwater-flow (GWF) model object. This creates a groundwater flow model and attaches it to the simulation
gwf = flopy.mf6.ModflowGwf(
    sim, # attaches the model to the simulation
    modelname=model_name, # assigns a name to the model
    model_nam_file=f"{model_name}.nam" # defines the name file used by MODFLOW to reference this model
)

## Step 3. Define the Spatial Grid (DIS)

MODFLOW solves the flow equation on a grid of **cells**. Each cell is a small, homogeneous block of aquifer with one value of head at its center (the *node*). The structured discretization package (DIS) defines:

| Variable | Meaning | Value in this model |
|---|---|---|
| `nlay`, `nrow`, `ncol` | number of layers, rows, and columns | 1 × 100 × 50 = 5,000 cells |
| `delr` | width of each **column**, measured along a row (the **x** direction) | W / ncol = 10,000 / 50 = 200 m |
| `delc` | height of each **row**, measured along a column (the **y** direction) | L / nrow = 20,000 / 100 = 200 m |
| `top` | elevation of the top of layer 1 (here, the land surface) | 100.05 to 109.95 m |
| `botm` | elevation of the bottom of each layer | `top − B` = 0.05 to 9.95 m |

**Conceptual model.** The domain is a 10 km (east–west) × 20 km (north–south) alluvial valley. It is represented as a single unconfined layer of thickness B = 100 m. The land surface slopes gently toward the south (slope = 0.0005, or 0.5 m per km). Top and bottom elevations therefore vary only with y and are constant along x.

**Grid indexing (important!).** Cells are identified by `(layer, row, column)`.
- FloPy uses **zero-based** indices: the first cell is `(0, 0, 0)`. MODFLOW input files use **one-based** indices: the same cell is `(1, 1, 1)`. FloPy converts between them automatically.
- **Row 0 is at the north (top) edge of the model** (y = 20,000 m), and rows increase toward the south. Column 0 is at the west (left) edge, and columns increase toward the east.
- This is why the cell-center coordinates are computed as `yc = maxy - (row + 0.5) * delc`.

> **Choosing a cell size.** Smaller cells resolve steep gradients (for example, near wells and rivers) more accurately, but they increase computation time. With 200 m cells, the head computed in a pumping-well cell is an *average for a 200 m × 200 m block*. It is not the water level you would measure inside the well casing.

In [ ]:
# We will use the following information from our conceptual model:

L = 20000.0  # Length of the model domain (meters)
W = 10000.0  # Width of the model domain (meters)
B = 100.0    # Thickness of the aquifer (meters)
slope_landscape = 0.0005  # Slope of the land surface (dimensionless)

# Grid dimensions (e.g., 20 km x 10 km region)
nlay = 1 # Number of layers
nrow = 100 # Number of rows
ncol = 50 # Number of columns
delr = W / ncol  # Cell width along rows (meters)
delc = L / nrow  # Cell width along columns (meters)

# Elevations

# We will create an elevation grid at cell centers for model top
minx = 0.0
maxy = L

xc = minx + (np.arange(ncol) + 0.5) * delr
yc = maxy - (np.arange(nrow) + 0.5) * delc

# Elevation varies with y and remains constant along x.
top = np.repeat((B + slope_landscape * yc)[:, np.newaxis], ncol, axis=1)
botm = top - B  # Aquifer of thickness B

dis = flopy.mf6.ModflowGwfdis(
    gwf,
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    delr=delr,
    delc=delc,
    top=top,
    botm=botm,
)

### Plot the Grid and the Top and Bottom Elevations

Always plot your model inputs before running a simulation. Most modeling errors (wrong units, flipped arrays, misplaced cells) are easy to see in a map.

- **Left panel:** the land surface (`top`), which increases from about 100 m in the south to about 110 m in the north.
- **Right panel:** the aquifer bottom (`botm`). It has the same pattern, shifted down by B = 100 m.

The thin gray lines are the 200 m × 200 m model cells.

In [ ]:
fig, axs = plt.subplots(1,2, figsize=(10, 5))
axs = axs.flatten()

ax = axs[0]
# Create a map view of top elevation
mv = PlotMapView(model=gwf, ax=ax)

# Plot the grid
mv.plot_grid(
    linewidth=0.25, 
    color='gray'
    )

# Plot an array (like top elevation) as colored cells
pc = mv.plot_array(
    top,
    masked_values=[],
    alpha=1.0,
    cmap='terrain',
    edgecolor='none',
    norm=None,
)


plt.colorbar(pc, ax=ax, label='Elevation (m)', shrink=0.7)
ax.set_title('Model grid with top elevation')
ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')

ax = axs[1]
# Create a map view of bottom elevation
mv = PlotMapView(model=gwf, ax=ax)

# Plot the grid
mv.plot_grid(
    linewidth=0.25, 
    color='gray'
    )

# Plot an array (like bottom elevation) as colored cells
pc = mv.plot_array(
    botm,
    masked_values=[],
    alpha=1.0,
    cmap='terrain',
    edgecolor='none',
    norm=None,
)


plt.colorbar(pc, ax=ax, label='Elevation (m)', shrink=0.7)
ax.set_title('Model grid with bottom elevation')
ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')

fig.tight_layout()

### Plot Cross Sections in the x Direction (West–East)

A cross section is a vertical slice through the grid. These three sections follow **rows** (constant y) at 25%, 50%, and 75% of the domain length. Elevations do not change in the x direction, so each section shows a flat layer 100 m thick.

Note the strong **vertical exaggeration**: the section is 10,000 m long but only about 100 m thick. Model cells are very thin compared with their horizontal size. This geometry is why horizontal flow dominates in regional aquifers and why a single-layer model (the *Dupuit approximation*) is often reasonable.

In [ ]:
r_cs = (np.array([0.25, 0.50, 0.75]) * nrow).astype(int) # Plot cross sections at 25%, 50%, and 75% of the model domain in the y-direction

fig, axes = plt.subplots(
    len(r_cs), 1, figsize=(5, 3 * len(r_cs)),
    sharex=True, constrained_layout=True
)
axes = np.atleast_1d(axes)

layer_ids = np.broadcast_to(
    np.arange(1, nlay + 1)[:, None, None],
    (nlay, nrow, ncol)
).astype(float)

layer_cmap = plt.cm.Set2(np.linspace(0, 1, nlay))

for ax, r0 in zip(axes, r_cs):
    xsec = flopy.plot.PlotCrossSection(model=gwf, ax=ax, line={"row": int(r0)})

    # plot ibound first so it does not hide layer colors
    xsec.plot_ibound(alpha=0.10)

    # plot all layers
    xsec.plot_array(
        layer_ids,
        cmap=plt.cm.colors.ListedColormap(layer_cmap),
        vmin=1,
        vmax=nlay,
        alpha=0.70
    )
    xsec.plot_grid(linewidth=0.3, color="0.6")

    # force full vertical extent so all layers are visible
    ax.set_ylim(np.nanmin(botm) - 5, np.nanmax(top) + 5)

    ax.set_title(f"Layer cross section at row {int(r0)} and y = {yc[int(r0)]:.1f} m")
    ax.set_ylabel("Elevation (m)")

axes[-1].set_xlabel("Distance along section (m)")
plt.show()

### Plot Cross Sections in the y Direction (North–South)

These three sections follow **columns** (constant x). The layer now slopes downward toward the south (left to right on the horizontal axis, because distance is measured from the north edge), following the land-surface slope of 0.0005. Over 20 km, the top and bottom drop by about 10 m.

In [ ]:
c_cs = (np.array([0.25, 0.50, 0.75]) * ncol).astype(int) # Plot cross sections at 25%, 50%, and 75% of the model domain in the x-direction

fig, axes = plt.subplots(
    len(c_cs), 1, figsize=(5, 3 * len(c_cs)),
    sharex=True, constrained_layout=True
)
if len(c_cs) == 1:
    axes = [axes]

layer_ids = np.broadcast_to(np.arange(1, nlay + 1)[:, None, None], (nlay, nrow, ncol)).astype(float)
layer_cmap = plt.cm.Set2(np.linspace(0, 1, nlay))

for ax, c0 in zip(axes, c_cs):
    xsec = flopy.plot.PlotCrossSection(model=gwf, ax=ax, line={"column": int(c0)})

    # plot ibound first so it does not hide layer colors
    xsec.plot_ibound(alpha=0.10)

    # plot all layers
    xsec.plot_array(
        layer_ids,
        cmap=plt.cm.colors.ListedColormap(layer_cmap),
        vmin=1, vmax=nlay, alpha=0.70
    )
    xsec.plot_grid(linewidth=0.3, color="0.6")

    # force full vertical extent so deeper layers are visible
    ax.set_ylim(np.nanmin(botm) - 5, np.nanmax(top) + 5)

    ax.set_title(f"Layer cross section at column {int(c0)} and x = {xc[int(c0)]:.1f} m")
    ax.set_ylabel("Elevation (m)")

axes[-1].set_xlabel("Distance along section (m)")
plt.show()

## Step 4. Aquifer Properties (NPF) and Initial Conditions (IC)

### Node Property Flow (NPF) package

The NPF package defines how easily water moves between cells. The key parameter is the **hydraulic conductivity** $K$. Alluvial aquifers consist of sand and gravel and are highly permeable. We use $K = 15$ m/d, which is typical of a medium to coarse sand. (Typical ranges: clean sand ≈ 1–100 m/d; gravel ≈ 100–1,000 m/d or higher; clay < 10⁻⁴ m/d.)

The option `icelltype` sets how the **saturated thickness** of each cell is calculated:

| `icelltype` | Behavior | Transmissivity |
|---|---|---|
| `0` | Confined: saturated thickness = full cell thickness | $T = K\,(z_{top} - z_{bot})$ (constant) |
| `1` | Convertible/unconfined: saturated thickness depends on the head | $T = K\,(h - z_{bot})$ when $h < z_{top}$ |

We use `icelltype=1` because the water table in an alluvial aquifer is free to rise and fall. The governing equation for steady, two-dimensional, unconfined flow (Dupuit approximation) is

$$
\frac{\partial}{\partial x}\left[K\,(h - z_{bot})\,\frac{\partial h}{\partial x}\right]
+ \frac{\partial}{\partial y}\left[K\,(h - z_{bot})\,\frac{\partial h}{\partial y}\right]
+ R + W = 0,
$$

where $h$ is hydraulic head [m], $R$ is recharge [m/d], and $W$ represents other sources and sinks per unit area (wells, river leakage) [m/d]. Transmissivity depends on $h$, so the equation is **nonlinear**. MODFLOW handles this through the outer iterations of the IMS solver.

**How MODFLOW solves this equation.** MODFLOW does not solve the differential equation directly. It writes a **water balance for every cell**: at steady state, the sum of all flows into and out of the cell must be zero. The flow between neighboring cells $i$ and $j$ follows Darcy's law written in terms of a *conductance* $C_{ij}$:

$$
Q_{ij} = C_{ij}\,(h_j - h_i), \qquad C_{ij} \approx \frac{K\,b\,\Delta w}{\Delta \ell},
$$

where $b$ is the saturated thickness, $\Delta w$ is the width of the shared cell face, and $\Delta \ell$ is the distance between cell centers. With one equation per cell, the result is a system of 5,000 equations for 5,000 unknown heads.

Two output options are also set:
- `save_flows=True` saves the cell-by-cell flows to the budget file. MODPATH needs these flows in Step 11.
- `save_specific_discharge=True` saves the Darcy flux vector $\mathbf{q} = -K\nabla h$ at each cell, which we use to draw flow arrows in Step 10.

### Initial Conditions (IC) package

The IC package sets the starting head in every cell (`strt`). In a **steady-state** model, the initial head is only the *starting guess* for the iterative solver. A reasonable value helps the solver converge, but it does not change the final answer. We use $h_0 = 95$ m, which lies between the aquifer bottom (≤ 10 m) and the land surface (≥ 100 m), so all cells start wet. In a **transient** model, the initial heads are the physical starting condition and strongly affect the results.

> **What about storage (STO)?** Specific yield $S_y$ and specific storage $S_s$ control how much water an aquifer releases when heads decline. They matter only for transient simulations. This model has no STO package, so MODFLOW 6 runs it as steady state.

In [ ]:
K = 15.0  # Hydraulic conductivity (m/day)
h_0 = 95.0  # Initial head (m)

# Node Property Flow (NPF) for flow properties
npf = flopy.mf6.ModflowGwfnpf(
    gwf,
    icelltype=1,      # 1 = Unconfined aquifer
    k=K,           # Hydraulic conductivity (m/day)
    save_flows=True,
    save_specific_discharge=True,
)

# Initial Heads
ic = flopy.mf6.ModflowGwfic(gwf, strt=h_0)

## Step 5. Boundary Conditions: River (RIV) and Constant Head (CHD)

Boundary conditions describe how the model domain exchanges water with its surroundings. They often control the solution more than any other input. This model uses three types:

| Boundary | Location | Mathematical type | MODFLOW package |
|---|---|---|---|
| River | East edge (last column, `ncol-1`) | Head-dependent flux (Cauchy / third type) | RIV |
| Regional groundwater levels | North row (row 0) and south row (row `nrow-1`) | Specified head (Dirichlet / first type) | CHD |
| Valley wall / impermeable bedrock | West edge (column 0) and the aquifer bottom | No flow (Neumann / second type, zero flux) | *none: this is the default* |

> **Key idea:** In MODFLOW, any cell face that is not assigned a boundary condition is a **no-flow boundary**. The west edge of this model acts as an impermeable valley wall because we did not assign anything there.

### The River package (RIV)

A river runs north to south along the east edge of the model, flowing downhill with the landscape (`river_slope = slope_landscape`). Each river cell is defined by `((layer, row, col), stage, conductance, rbot)`:

- **Stage** $h_{riv}$: elevation of the river water surface [m]. Here, the stage is set equal to the land surface at the river cell (bankfull conditions). It is highest at the upstream (north) end and decreases linearly downstream with the land-surface slope.
- **Conductance** $C_{riv}$ [m²/d]: describes how easily water passes through the riverbed sediments:
  $$C_{riv} = \frac{K_{bed}\, L_{r}\, W_{r}}{M},$$
  where $K_{bed}$ is the riverbed hydraulic conductivity, $L_r$ is the length of river within the cell, $W_r$ is the river width, and $M$ is the riverbed thickness. For example, $K_{bed} = 0.5$ m/d, $L_r = 200$ m, $W_r = 5$ m, and $M = 1$ m give $C_{riv} = 500$ m²/d, the value used below.
- **Rbot** $z_{rbot}$: elevation of the bottom of the riverbed sediments [m]. Here it is set `river_depth` = 2 m below the stage, so the river channel is incised 2 m into the land surface.

MODFLOW computes the flow between the river and the aquifer cell as

$$
Q_{riv} =
\begin{cases}
C_{riv}\,(h_{riv} - h), & h > z_{rbot}\\[4pt]
C_{riv}\,(h_{riv} - z_{rbot}), & h \le z_{rbot}
\end{cases}
$$

- If $Q_{riv} > 0$, water flows **from the river into the aquifer**: a **losing** reach.
- If $Q_{riv} < 0$, water flows **from the aquifer into the river**: a **gaining** reach.
- When the water table drops below the riverbed ($h \le z_{rbot}$), the river and aquifer become *disconnected*. Leakage then reaches a maximum and no longer increases as the water table falls.

### The Constant-Head package (CHD)

Along the northern and southern rows, we fix the head to represent regional groundwater levels that the local aquifer cannot change. Each CHD row is set equal to the river stage at that end of the valley:
- North row (row 0, upstream): $h$ = upstream river stage (109.95 m)
- South row (row `nrow-1`, downstream): $h$ = downstream river stage (100.05 m)

The resulting regional gradient drives groundwater from north to south, parallel to the river. A CHD cell can supply or remove **any** amount of water needed to keep its head fixed. This is a strong assumption, so place CHD boundaries far from the area of interest (for example, from pumping wells) whenever possible.

In [ ]:
river_slope = slope_landscape  # m/m (drop per meter downstream). For simplicity, we will assume that the river slope is equal to the landscape slope. This is a reasonable assumption for a river that is in equilibrium with the surrounding landscape.
river_depth = 2.0           # depth of the river channel below the land surface (m)
cond = 500.0                # Conductance of river sediment (m^2/day)

# The river occupies the last (eastern) column. Its stage is set at the land surface
# (bankfull conditions) and its bed lies river_depth below the land surface.
riv_col = ncol - 1
stage_upstream = top[0, riv_col]                                     # stage at row 0 (north, upstream) (m)
stage_downstream = stage_upstream - river_slope * ((nrow-1) * delc)  # stage at last row (south, downstream) (m)

print(f"River stage at downstream end: {stage_downstream:.2f} m")
print(f"River stage at upstream end: {stage_upstream:.2f} m")

riv_data = []


for r in range(nrow):
    y = r * delc            # distance downstream from row 0 (m), since river runs by row
    stage = stage_upstream - river_slope *  y
    rbot = stage - river_depth
    riv_data.append(((0, r, riv_col), stage, cond, rbot))

# Initialize RIV package
riv = flopy.mf6.ModflowGwfriv(
    gwf,
    pname="riv",
    maxbound=len(riv_data),
    stress_period_data={0: riv_data},
    save_flows=True
)

# Impose a constant head boundary on the top and bottom rows to simulate regional flow
chd_data = []
for c in range(ncol):
    chd_data.append(((0, 0, c), riv_data[0][1]))   # Top row 
    chd_data.append(((0, nrow-1, c), riv_data[-1][1]))  # Bottom row  

chd = flopy.mf6.ModflowGwfchd(
    gwf,
    pname="chd",
    maxbound=len(chd_data),
    stress_period_data={0: chd_data},
    save_flows=True
)


### Check the River Boundary

Before running the model, verify that the river is in the right place and that its stage decreases downstream as intended. The map below colors only the RIV cells (the eastern column) by their stage: highest (≈ 110 m) at the north end and lowest (≈ 100 m) at the south end.

In [ ]:
# Build a 2D array with river stage values at RIV cells (layer 1 / index 0)
riv_stage = np.full((nrow, ncol), np.nan, dtype=float)
for (k, r, c), stage, _, _ in riv_data:
    if k == 0:
        riv_stage[r, c] = stage

fig, ax = plt.subplots(figsize=(5, 10))
mv = PlotMapView(model=gwf, ax=ax, layer=0)

# Plot stage only where RIV is assigned
pc = mv.plot_array(
    np.ma.masked_invalid(riv_stage),
    cmap="viridis",
    alpha=0.95
)

# Plot mesh
mv.plot_grid(linewidth=0.3, color="gray", alpha=0.6)

# Optional context
# river_areas_gdf_xy_ready.boundary.plot(ax=ax, color="black", linewidth=1)

plt.colorbar(pc, ax=ax, shrink=0.75, label="RIV stage (m)")
ax.set_title("Model mesh and RIV boundary stage elevations")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_aspect("equal")
plt.show()

# fig.set_tight_layout()

## Step 6. Adding Regional Recharge (RCH)

Alluvial aquifers are replenished by precipitation that infiltrates and percolates down to the water table. The **recharge** package adds a specified flux of water (length/time) to every cell of the uppermost active layer. `ModflowGwfrcha` is the *array-based* version of the package, which assigns recharge as a map, here a single uniform value.

| Quantity | Value |
|---|---|
| Recharge rate $R$ | 0.0001 m/d = 0.1 mm/d ≈ 36.5 mm/yr |
| Model area | 10 km × 20 km = 2 × 10⁸ m² |
| Total recharge volume (if active) | $R \times$ area = 20,000 m³/d |

> **Note:** In the cell below, the recharge rate is multiplied by `0`, so **recharge is switched off** in the base case. The water entering the aquifer then comes only from the river and the northern CHD boundary. To activate recharge, remove the `* 0`. Compare the total recharge (20,000 m³/d) with the pumping rate of a single well (10,000 m³/d): how much might recharge change the source of the water captured by the well?

In [ ]:
recharge_rate = 0.0001 # m/day (36.5 mm/year)

rch = flopy.mf6.ModflowGwfrcha(
    gwf, 
    recharge=recharge_rate  * 0
)

## Step 7. Adding Wells (WEL)

The WEL package specifies a **volumetric flow rate** $Q$ [m³/d] in selected cells. This is a *specified-flux* boundary. MODFLOW adds or removes exactly this amount of water, regardless of the head.

**Sign convention (same for all MODFLOW packages):**
- $Q < 0$: water is **removed** from the aquifer (pumping/extraction well)
- $Q > 0$: water is **added** to the aquifer (injection well)

Each well entry has the format `((layer, row, col), rate)`:

| Well | Cell (layer, row, col) | Approx. location (x, y) | Rate (m³/d) | Status in base case |
|---|---|---|---|---|
| Well 1 | (0, 25, 40) | (8,100 m, 14,900 m) | −10,000 (pumping) | **off** (`* 0`) |
| Well 2 | (0, 10, 30) | (6,100 m, 17,900 m) | +8,000 (injection) | **off** (`* 0`) |
| Well 3 | (0, 50, 25) | (5,100 m, 9,900 m) | −10,000 (pumping) | **on** |

For scale, 10,000 m³/d ≈ 116 L/s ≈ 1,800 gal/min, comparable to a large municipal or irrigation well (or a small well field).

> **Note:** MODFLOW spreads the well's withdrawal over the entire 200 m × 200 m cell. The computed head in a well cell therefore underestimates the drawdown inside the actual well bore, which has a radius of only about 0.1–0.3 m.

To turn Wells 1 and 2 on, remove the `* 0` from their rates.

In [ ]:
# -------------------------------------------------------------
# Define Well Data
# Format: ((layer, row, col), rate)  -> rate in m^3/day
# -------------------------------------------------------------
well_data = [
    # Well 1: Pumping 
    ((0, 25, 40), -10000.0 * 0), 
    
    # Well 2: An injection well recharging the aquifer  
    ((0, 10, 30), 8000.0 * 0),
    
    # Well 3: Pumping
    ((0, 50, 25), -10000.0 )
]

# Initialize the WEL package
wel = flopy.mf6.ModflowGwfwel(
    gwf,
    pname="wel",
    maxbound=len(well_data),
    stress_period_data={0: well_data},
    save_flows=True  # Important so we can see local flow vectors change!
)

## Step 8. Output Control (OC)

The OC package controls **what** MODFLOW saves and **when**. We request two binary output files:

| File | Contents | Used for |
|---|---|---|
| `alluvial_model.hds` | Hydraulic head in every cell | Head maps, profiles, depth to water |
| `alluvial_model.cbc` | Cell-by-cell budget: flows between cells and to/from every boundary (RIV, CHD, WEL, RCH), plus specific discharge | Flow vectors, river leakage, particle tracking |

`saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")]` saves heads and budgets for **all** time steps. (We have only one.)

MODFLOW also always writes a text **listing file** (`alluvial_model.lst`). It contains a summary of the input, the solver's progress, and the **volumetric water budget**. The budget lists every inflow and outflow and the *percent discrepancy* between total inflow and total outflow. A good solution has a percent discrepancy close to 0 (well below 1%).

In [ ]:
head_file = f"{model_name}.hds"
budget_file = f"{model_name}.cbc"

oc = flopy.mf6.ModflowGwfoc(
    gwf,
    head_filerecord=head_file,
    budget_filerecord=budget_file,
    saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")]
)

## Step 9. Write the Input Files and Run the Model

Two commands:

1. `sim.write_simulation()` writes one text input file per package into `workspace_alluvial_model/` (for example `alluvial_model.dis`, `alluvial_model.npf`, and `riv.riv`). These are ordinary text files. Open a few to see exactly what MODFLOW reads.
2. `sim.run_simulation()` calls the `mf6` executable, which reads those files, solves the equations, and writes the output files.

`success` is `True` if MODFLOW reports *"Normal termination of simulation."* If the run fails:
- check that `MF6_EXE` points to a valid executable (Step 0);
- read the end of the listing file (`alluvial_model.lst`) or `mfsim.lst` for error messages;
- check for convergence failures (for example, cells going dry or unrealistic K values or rates).

In [ ]:
sim.write_simulation()
success, buff = sim.run_simulation()

if not success:
    raise RuntimeError("MODFLOW 6 did not run successfully.")

## Step 10. Post-Processing: Plotting the Results of the Flow Simulation

After the simulation runs successfully, we read the binary output and visualize the results. The next cell:

1. Reads the **heads** from `alluvial_model.hds`. `heads` has shape `(nlay, nrow, ncol)`.
2. Reads the **budget file** `alluvial_model.cbc` and extracts the specific discharge (`DATA-SPDIS`).
3. Converts it to the components $q_x$, $q_y$, $q_z$ at the cell centers.
4. Builds the contour levels for the head maps, every `head_step` = 1 m.

> **Specific discharge vs. velocity.** The specific discharge (Darcy flux) $\mathbf{q} = -K\nabla h$ [m/d] is the volume of water that crosses a unit area of aquifer (solids + pores) per unit time. Water actually moves only through the connected pores, so the average **seepage (linear) velocity** is faster:
> $$\mathbf{v} = \frac{\mathbf{q}}{n_e},$$
> where $n_e$ is the effective porosity. MODPATH uses $\mathbf{v}$ to move particles in Step 11.

In [ ]:
# Load the binary head file
hds = flopy.utils.binaryfile.HeadFile(os.path.join(model_dir, head_file))
heads = hds.get_data()
cbc = flopy.utils.binaryfile.CellBudgetFile(os.path.join(model_dir, budget_file))

spdis = cbc.get_data(text="DATA-SPDIS")[0]
qx, qy, qz = flopy.utils.postprocessing.get_specific_discharge(
    spdis,
    gwf,
    head=heads
)

head_min = heads.flatten().min()
head_max = heads.flatten().max()
head_step = 1 # Step size for contour levels (m)
head_range = np.arange(np.ceil(head_min*10)/10, np.floor(head_max*10)/10, head_step)
print(f"Head range: {head_min:.2f} m to {head_max:.2f} m")

### 10.1 Head Contours and Flow Vectors

This map combines several layers of information:
- **blue lines:** contours of hydraulic head (the water table), labeled in meters;
- **cyan cells:** river (RIV) cells along the eastern edge;
- **red cells:** well (WEL) cells. FloPy marks every cell listed in the WEL package, including wells whose rate is set to zero;
- **black arrows:** specific discharge $\mathbf{q}$, plotted every second cell (`istep=2`, `jstep=2`). With `normalize=False`, arrow length is proportional to the magnitude of $\mathbf{q}$, so faster flow appears as longer arrows.

**How to read it:**
- Groundwater flows from high head to low head, **perpendicular to the head contours** (in an isotropic aquifer).
- Far from the well, contours run nearly west–east and flow is southward, driven by the regional gradient between the two CHD boundaries.
- Around the pumping well, the contours bend into a **cone of depression**: heads drop and flow converges radially toward the well.
- Near the river, arrows pointing *toward* the river indicate groundwater discharge (gaining reach). Arrows pointing *away* from it indicate river water entering the aquifer (losing reach).

In [ ]:
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(1, 1, 1, aspect="equal")

# modelmap = flopy.plot.PlotMapView(model=gwf, ax=ax)
pmv = flopy.plot.PlotMapView(model=gwf, ax=ax)

# 1. Plot the grid and boundary conditions (River in Cyan, Wells in Red)
pmv.plot_ibound()
pmv.plot_grid(colors="0.5", lw=0.3)      # mesh lines
pmv.plot_bc("RIV", color="cyan")
pmv.plot_bc("WEL", color="red")  # This automatically plots your well locations

# 2. Plot head contours
contours = pmv.contour_array(heads[0], colors="blue", levels=head_range)
plt.clabel(contours, inline=True, fmt="%1.1f m", fontsize=7)

# 3. Plot the flow vectors
# We turn normalize=False here so you can visually see the groundwater 
# accelerate (longer arrows) as it gets sucked into the pumping wells.

vectors = pmv.plot_vector(
    qx, 
    qy, 
    istep=2, 
    jstep=2, 
    normalize=False,   
    color="black", 
    scale=.5,         # You may need to tweak the scale if arrows are too long/short
    headwidth=3, 
    headlength=4
)

ax.set_title("Impact of Pumping Wells on Regional Alluvial Aquifer Flow")
ax.set_xlabel("Distance X (m)")
ax.set_ylabel("Distance Y (m)")
plt.show()

### 10.2 Head Profile Along a Column (North–South)

The next cell extracts the heads along a single column (`the_col`) and plots them against distance y. The red dots show the river stage at the upstream and downstream ends, which are also the heads imposed by the CHD boundaries.

`the_col = 25` passes through pumping Well 3 (column 25), so the profile crosses the **cone of depression**. Without the well, the profile would decrease almost linearly from 109.95 m to 100.05 m. The dip in the middle is the drawdown caused by pumping.

> **Try it:** Set `the_col = ncol - 1` to plot the heads in the river column. Set `the_col = 0` to plot them along the western no-flow boundary. How do the shapes differ?

In [ ]:
# Head profile along y for a single column (the_col)

the_col = 25  # column 25 passes through Well 3; use ncol - 1 for the river column

head_col = heads[0, :, the_col]  # Extract head values along the selected column
y_vals = yc                      # y-coordinate of each cell center (m); row 0 is at the north edge

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(y_vals, head_col, marker="o", linewidth=1.5, markersize=4, label="Hydraulic head")
y_upstream = yc[0]          # northern (upstream) end
y_downstream = yc[nrow-1]   # southern (downstream) end
ax.scatter([y_upstream, y_downstream], [stage_upstream, stage_downstream], color="red", s=50, zorder=5, label="River stage at the ends")  # Highlight points with red dots
ax.set_xlabel("Y distance (m)")
ax.set_ylabel("Head (m)")
ax.set_title(f"Hydraulic Head Along Column {the_col} (x = {xc[the_col]:.0f} m)")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()

### 10.3 Head Profiles Along the Boundary Rows (West–East)

This cell plots the heads along the first row (row 0, the **northern/upstream** CHD boundary) and the last row (row `nrow-1`, the **southern/downstream** CHD boundary). Both rows are constant-head cells, so the heads along each should be **perfectly flat** and equal to the values we imposed (109.95 m and 100.05 m). The red dots mark the river stage at the two ends of the river (in the last column, x = 9,900 m).

This plot is a simple **check** that the boundary conditions were applied correctly. To see how heads vary from the valley wall to the river, change the row index to an interior row, for example `heads[0, 50, :]`, which passes through Well 3.

In [ ]:
# Head profiles along x for the first and last rows (the CHD boundaries)
head_row_upstream = heads[0, 0, :]          # Row 0: northern (upstream) boundary
head_row_downstream = heads[0, nrow-1, :]   # Last row: southern (downstream) boundary

x_vals = xc  # x-coordinate of each cell center (m)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x_vals, head_row_upstream, marker="o", linewidth=1.5, label="Upstream (north) row")
ax.plot(x_vals, head_row_downstream, marker="s", linewidth=1.5, label="Downstream (south) row")
x_river = xc[riv_col]  # the river is in the last column at both ends
ax.scatter([x_river, x_river], [stage_upstream, stage_downstream], color="red", zorder=5, label="River stage")  # Highlight points with red dots
ax.set_xlabel("X distance (m)")
ax.set_ylabel("Head (m)")
ax.set_title("Hydraulic Head Along the Boundary Rows")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()

### 10.4 Depth to the Water Table

The depth to the water table is the distance from the land surface down to the water table:

$$\text{depth} = z_{top} - h.$$

In a single-layer unconfined model, the water table is simply the computed head. This map matters in practice: it controls the depth a well must be drilled to, the risk of shallow contamination, and whether vegetation can reach groundwater.

Where the computed head is **at or above** the land surface, the depth is set to 0. In reality, such cells would discharge water as springs, seeps, or wetlands. To represent that process, a more complete model would add a drain (DRN) boundary at the land surface. Look for these zones near the boundaries and the river.

In [ ]:
# Water table depth (m) = land surface (top) - water table elevation
if "wt_elev" in globals():
    wt = wt_elev
else:
    hmask = np.ma.masked_where(heads < 0, heads)
    wt = np.nanmax(hmask, axis=0)

wt_depth = top - wt
wt_depth = np.where(np.isfinite(wt_depth), np.maximum(wt_depth, 0.0), np.nan)

wt_depth_ma = np.ma.masked_invalid(wt_depth)
levels = np.arange(0, np.max(wt_depth) + 1, 0.5)

fig, ax = plt.subplots(figsize=(5, 10))
pmv = PlotMapView(model=gwf, ax=ax, layer=0)

# Optional background
pc = pmv.plot_array(wt_depth_ma, cmap="plasma", alpha=0.75)

# Define color range (clip to robust upper bound)
vmin = 0.0
vmax = np.max(wt_depth)
pc.set_clim(vmin=vmin, vmax=vmax)
# Contour lines
cs = pmv.contour_array(wt_depth_ma, levels=levels, colors="black", linewidths=0.7)
ax.clabel(cs, fmt="%.1f", fontsize=7)

pmv.plot_grid(linewidth=0.2, color="0.7", alpha=0.4)


plt.colorbar(pc, ax=ax, shrink=0.75, label="Water table depth (m)")
ax.set_title("Water table depth contours")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_aspect("equal")
plt.show()

### 10.5 Gaining and Losing River Reaches

For each river cell, we compare the aquifer head $h$ with the river stage $h_{riv}$ (see the RIV equation in Step 5):

| Condition | Direction of flow | Reach type | Color |
|---|---|---|---|
| $h > h_{riv}$ | aquifer → river | **Gaining** | blue |
| $h < h_{riv}$ | river → aquifer | **Losing** | red |
| $h \approx h_{riv}$ (within `tol` = 1 mm) | negligible exchange | Neutral | white |

Under natural conditions, rivers in humid regions are usually gaining. Pumping near a river lowers the water table and can turn a gaining reach into a losing one. Compare this map with the capture-zone results in Step 11.

In [ ]:
# Map of gaining/losing surface-water exchange at RIV cells
# Convention:
#   +1  -> gaining stream (GW -> river; aquifer head > river stage)
#   -1  -> losing stream  (river -> GW; aquifer head < river stage)
#    0  -> near-neutral

tol = 1e-3  # m
riv_exchange = np.full((nrow, ncol), np.nan, dtype=float)

for (k, r, c), stage, _, _ in riv_data:
    h = heads[k, r, c]
    if not np.isfinite(h) or h < 0:
        continue
    if h > stage + tol:
        riv_exchange[r, c] = 1.0
    elif h < stage - tol:
        riv_exchange[r, c] = -1.0
    else:
        riv_exchange[r, c] = 0.0

fig, ax = plt.subplots(figsize=(5, 10))
pmv = PlotMapView(model=gwf, ax=ax, layer=0)

arr = np.ma.masked_invalid(riv_exchange)
pc = pmv.plot_array(arr, cmap="bwr_r", vmin=-1, vmax=1, alpha=0.95)
pmv.plot_grid(linewidth=0.2, color="0.75", alpha=0.5)



cbar = plt.colorbar(pc, ax=ax, shrink=0.75, ticks=[-1, 0, 1])
cbar.ax.set_yticklabels(["Losing (river → GW)", "Neutral", "Gaining (GW → river)"])

# Legend proxies
ax.scatter([], [], c="red", label="Losing reach")
ax.scatter([], [], c="white", edgecolor="k", label="Neutral reach")
ax.scatter([], [], c="blue", label="Gaining reach")
ax.legend(loc="lower left", frameon=True)

ax.set_title("Gaining and Losing Surface-Water Reaches (RIV cells)")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_aspect("equal")
plt.show()

### 10.6 Inspect the Model Packages

`gwf.packagelist` prints every package in the model together with its input data. Use it to confirm that the model contains exactly what you intended, which is good practice before trusting any results.

In [ ]:
print(gwf.packagelist)

## Step 11. Capture Zone for Each Pumping Well (MODPATH 7)

### What is a capture zone?

The **capture zone** (or *contributing area*) of a well is the part of the aquifer, and of its boundaries, from which water eventually flows to that well. Capture zones are central to **wellhead protection**. A contaminant released inside the capture zone will eventually reach the well; one released outside will not.

### How MODPATH works

MODPATH 7 is a **particle-tracking** program. It does not solve any flow equation. Instead, it reads the heads and the cell-by-cell flows computed by MODFLOW and moves imaginary, massless "water particles" with the **seepage velocity** $\mathbf{v} = \mathbf{q}/n_e$. Inside each cell, MODPATH assumes that each velocity component varies linearly between opposite cell faces, which lets it calculate pathlines exactly within each cell (*Pollock's semi-analytical method*). Particles move by **advection only**: dispersion, diffusion, and chemical reactions are ignored.

- **Forward tracking** starts particles at a source and follows them *downgradient* in the direction of flow ("where does this water go?").
- **Backward tracking** starts particles at a sink, such as a well, and follows them *upgradient* against the flow ("where did this water come from?"). The region covered by the backward pathlines from a well approximates its capture zone.

### What the next cell does

1. **Initialize MODPATH 7.** `Modpath7` links MODPATH to the MODFLOW flow model (`gwf`) and to the model workspace, so it can read the flow results (`.hds`, `.cbc`, and grid files).
2. **Set basic properties.** `Modpath7Bas` assigns an effective porosity $n_e = 0.25$. Porosity controls particle **travel times** ($v = q/n_e$). In a steady-state flow field it does **not** change the shape of the pathlines.
3. **Place particles at the pumping wells.** The loop selects the wells with a negative (pumping) rate, skipping inactive and injection wells, and converts each well's `(layer, row, column)` location to a single global *node number* (MODPATH identifies cells by node number). Each well cell is divided into a 10 × 10 × 1 array of starting locations, giving **100 particles per well**.
4. **Configure the tracking simulation.** Pathlines are computed with `trackingdirection="backward"`. The options `weaksinkoption="stop_at"` and `weaksourceoption="stop_at"` stop particles when they enter a *weak sink* or *weak source* cell. A **weak sink** is a cell where the sink (for example, a well) removes only *part* of the water flowing through the cell, so some water passes through. Stopping particles there is the conservative choice for delineating capture zones. Particles also stop when they reach a boundary that supplies water, such as a CHD cell or a losing river cell.
5. **Write and run.** `mp.write_input()` writes the MODPATH input files and `mp.run_model()` runs the particle tracking. The output summarizes how many particles ended at each type of boundary or source/sink cell. `(True, [])` indicates the run finished successfully.

The resulting pathlines are plotted in the next cells.

In [ ]:
# Name for the MODPATH files
mp_name = f"{model_name}_mp"

# -------------------------------------------------------------
# 1. Initialize the MODPATH 7 simulation
# -------------------------------------------------------------

mp = flopy.modpath.Modpath7(
    modelname=mp_name, 
    flowmodel=gwf, 
    exe_name=MP7_EXE, 
    model_ws=model_dir
)

# -------------------------------------------------------------
# 2. Set up the MODPATH execution options
# -------------------------------------------------------------

# we use 'backwards' tracking to see where the well's water originates!
mpbas = flopy.modpath.Modpath7Bas(mp, porosity=0.25)

# -------------------------------------------------------------
# 3. Create particles around the wells (FIXED)
# -------------------------------------------------------------
node_numbers = []
well_ids = []  # Well number (1, 2, 3, ...) as listed in well_data, used for plot labels
for iw, well in enumerate(well_data):
    if well[1] < -1e-6:  # Include only pumping wells (negative rate); skip inactive and injection wells
        print(f"Creating particles for well at layer {well[0][0]}, row {well[0][1]}, col {well[0][2]} with rate {well[1]} m^3/day")
        # Get the global node number for the well cell
        layer, row, col = well[0]
        node = gwf.modelgrid.get_node([(layer, row, col)])[0]
        node_numbers.append(node)
        well_ids.append(iw + 1)

# Set up the cell data type template to place an array of particles in the cell
# 10x10x1 means 100 particles distributed inside each well cell
subdivision_data = flopy.modpath.CellDataType(
    drape=0, 
    columncelldivisions=10, 
    rowcelldivisions=10, 
    layercelldivisions=1
)

# Initialize the node particle data object
node_particle_data = flopy.modpath.NodeParticleData(
    subdivisiondata=subdivision_data,
    nodes=np.asarray(node_numbers, dtype=int),
)

# Wrap it into a Particle Group Node Template
pg = flopy.modpath.ParticleGroupNodeTemplate(
    particlegroupname="well_particles", 
    particledata=node_particle_data
)

# -------------------------------------------------------------
# 4. Configure the tracking simulation
# -------------------------------------------------------------
mpsim = flopy.modpath.Modpath7Sim(
    mp,
    simulationtype="pathline",
    trackingdirection="backward",  # Track backward from the well
    weaksinkoption="stop_at",      # Tells particles to stop when they reach a well
    weaksourceoption="stop_at",
    budgetoutputoption="summary",
    particlegroups=pg,
)

# -------------------------------------------------------------
# 5. Write input files and run MODPATH
# -------------------------------------------------------------
mp.write_input()
mp.run_model()

### Plot the Results of the Particle Tracking

The next cell plots the backward pathlines from each active pumping well on top of the head contours, using a different color for each well. Each pathline starts in the well cell and ends where its water *entered* the aquifer, either at the river or at the northern CHD boundary (or at the water table, if recharge is active).

What to look for:
- Pathlines cross the head contours at right angles, because $K$ is isotropic.
- The outer envelope of the pathlines outlines the **capture zone**. It extends upgradient (north) because of the regional gradient and widens where the well draws water from the river.
- Downgradient of the well there is a **stagnation point**, where the well's pull exactly balances the regional flow. Water south of this point is not captured.

In [ ]:
# 1. Load the pathline data
pathline_file = os.path.join(model_dir, f"{mp_name}.mppth")
p = flopy.utils.PathlineFile(pathline_file)

# Define a list of distinct colors for your wells
# (Add more colors if you add more wells later!)
capture_zone_colors = ["red", "darkorange", "purple", "green", "magenta"]

# 2. Set up the Map View
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(1, 1, 1, aspect="equal")
modelmap = flopy.plot.PlotMapView(model=gwf, ax=ax)

# Plot standard features
modelmap.plot_ibound()
modelmap.plot_bc("RIV", color="cyan")

# 1. This tints the well cell background subtly so you know which cell it is
modelmap.plot_bc("WEL", color="lightgray")

# Plot Head Contours
contours = modelmap.contour_array(heads[0], colors="blue", levels=head_range)
plt.clabel(contours, inline=True, fmt="%1.1f m", fontsize=10)

# Loop through each well and plot its pathlines with a unique color
for i, node in enumerate(node_numbers):
    # Extract pathlines strictly bound for this specific well node
    well_pathlines = p.get_destination_pathline_data(node)
    
    # Pick a color from our list (wrap around using modulo if there are more wells than colors)
    color = capture_zone_colors[i % len(capture_zone_colors)]
    
    # Plot this specific well's pathlines
    # (Note: travel_time parameter is omitted entirely to avoid parsing errors)
    modelmap.plot_pathline(
        well_pathlines, 
        colors= color, 
        linewidth=1.5, 
        alpha=0.4,
        label=f"Well {well_ids[i]} Capture Zone"
    )

# Add a legend to easily identify which zone belongs to which well
ax.legend(loc="lower left", fontsize=12, framealpha=0.9)

ax.set_title("Delineated Well Capture Zones (Multi-Color Tracking)")
ax.set_xlabel("Distance X (m)")
ax.set_ylabel("Distance Y (m)")

### Capture Zone by Source of Water

A key question for any well near a river is: **how much of the pumped water is actually river water?** Pumping lowers the water table near the river and can reverse the local gradient. Water is then *induced* to infiltrate from the river into the aquifer, a process known as **induced infiltration** or **riverbank filtration**. This matters for water quality (river contaminants can reach the well) and for water rights (pumping groundwater depletes streamflow).

In backward tracking, the **last point** of each pathline is where the water entered the aquifer. The next cell classifies every pathline by that endpoint:
- **blue:** the pathline ends in the river column (the water came from the river);
- **well color:** the pathline ends elsewhere, mostly at the northern CHD boundary (regional groundwater inflow) or, when recharge is active, at the water table.

This is an approximate, particle-count-based classification. The particles are evenly spaced in the well cell but do not carry equal fractions of the pumped flow. For a precise volumetric split, compare the RIV and CHD terms in the water budget.

In [ ]:
# Load the pathline data
pathline_file = os.path.join(model_dir, f"{mp_name}.mppth")
p = flopy.utils.PathlineFile(pathline_file)

capture_zone_colors = ["red", "darkorange", "purple", "green", "magenta"]
river_origin_color = "blue"

fig, ax = plt.subplots(figsize=(10, 10))
ax.set_aspect("equal")
modelmap = flopy.plot.PlotMapView(model=gwf, ax=ax)

modelmap.plot_ibound()
modelmap.plot_bc("RIV", color="cyan")
modelmap.plot_bc("WEL", color="lightgray")

contours = modelmap.contour_array(heads[0], colors="blue", levels=head_range)
plt.clabel(contours, inline=True, fmt="%1.1f m", fontsize=10)

legend_handles = []

for i, node in enumerate(node_numbers):
    well_color = capture_zone_colors[i % len(capture_zone_colors)]
    well_pathlines = p.get_destination_pathline_data(node)

    for pathline in well_pathlines:
        # In backward tracking, the final point indicates the particle's source.
        x_end = pathline["x"][-1]
        y_end = pathline["y"][-1]

        ends_at_river = (
            (ncol - 1) * delr <= x_end <= ncol * delr
            and 0 <= y_end <= L
        )
        color = river_origin_color if ends_at_river else well_color

        modelmap.plot_pathline(
            pathline,
            colors=color,
            linewidth=1.5,
            alpha=0.6,
        )

    legend_handles.append(
        ax.plot([], [], color=well_color, label=f"Well {well_ids[i]}: other sources")[0]
    )

legend_handles.append(
    ax.plot([], [], color=river_origin_color, label="River source")[0]
)
ax.legend(handles=legend_handles, loc="lower left", fontsize=12, framealpha=0.9)

ax.set_title("Well Capture Zones (River Sources in Blue)")
ax.set_xlabel("Distance X (m)")
ax.set_ylabel("Distance Y (m)")
plt.show()

## Summary

In this notebook, we:

1. Built a conceptual model of a regional alluvial aquifer bounded by a river, regional constant-head boundaries, and an impermeable valley wall.
2. Translated it into a MODFLOW 6 model with FloPy, one package at a time (TDIS, IMS, DIS, NPF, IC, RIV, CHD, RCH, WEL, OC).
3. Solved for steady-state heads and examined the water table, flow directions, depth to water, and river–aquifer exchange.
4. Used MODPATH 7 backward particle tracking to delineate the capture zone of a pumping well and to identify how much of its water comes from the river.

The same workflow applies to every MODFLOW project, from classroom examples to regional water-management models: **conceptualize → discretize → assign properties and boundaries → run → check → interpret**.

## Exercises

Change one input at a time, re-run the notebook (**Kernel → Restart & Run All**), and describe what changes and why.

1. **Back-of-the-envelope check.** With no wells and no recharge, the regional hydraulic gradient is about $(109.95 - 100.05)/19{,}800 \approx 0.0005$.
   - Compute the specific discharge $q = K\,i$ and the seepage velocity $v = q/n_e$. How many years would water take to travel the 20 km length of the valley?
   - Estimate the regional groundwater flow through the valley, $Q \approx K\,b\,i\,W$ (with $b \approx 100$ m and $W$ = 10 km). Compare it with the pumping rate of Well 3. What does this tell you about where the well's water must come from?
2. **Recharge.** Activate recharge (remove `* 0` in Step 6). How do the heads, the depth to the water table, and the gaining/losing reaches change? How does the fraction of river water captured by Well 3 change?
3. **More wells.** Activate Wells 1 and 2 (Step 7). How does the injection well (Well 2) affect the capture zone of Well 1? Do the capture zones of the pumping wells interact?
4. **Pumping rate.** Double the pumping rate of Well 3. Does the capture zone grow? Does the river become losing over a longer reach?
5. **Hydraulic conductivity.** Repeat the base case with $K$ = 1.5 m/d and $K$ = 150 m/d. Explain the change in drawdown at the well.
6. **Riverbed conductance.** Reduce `cond` from 500 to 5 m²/d (a clogged, silty riverbed). How does the river–aquifer connection, and the source of water to the well, change?
7. **Porosity.** Change the porosity in MODPATH from 0.25 to 0.10. Do the pathlines change? What does change? *(Hint: think about $v = q/n_e$.)*
8. **Conceptual thinking.** The west edge is a no-flow boundary. What would change if it were a second river? If it were a constant-head boundary?

## Further Reading

- Langevin, C.D., Hughes, J.D., Banta, E.R., Niswonger, R.G., Panday, S., and Provost, A.M., 2017, *Documentation for the MODFLOW 6 Groundwater Flow Model*: U.S. Geological Survey Techniques and Methods, book 6, chap. A55.
- Pollock, D.W., 2016, *User guide for MODPATH Version 7—A particle-tracking model for MODFLOW*: U.S. Geological Survey Open-File Report 2016–1086.
- Bakker, M., Post, V., Langevin, C.D., Hughes, J.D., White, J.T., Starn, J.J., and Fienen, M.N., 2016, Scripting MODFLOW model development using Python and FloPy: *Groundwater*, 54(5), 733–739.
- Anderson, M.P., Woessner, W.W., and Hunt, R.J., 2015, *Applied Groundwater Modeling: Simulation of Flow and Advective Transport*, 2nd ed.: Academic Press.
- FloPy documentation and examples: https://flopy.readthedocs.io
- MODFLOW 6 documentation: https://modflow6.readthedocs.io